# scikit-verify: see the math your code computes

You write numerical code. The computer runs it and gives you numbers. But the numbers hide something: the *formula* your code actually computed.

`to_sympy(fn, args)` runs your function once and hands you that formula, written in sympy. You change nothing about your code.

Let's build up from the simplest possible example.

In [1]:
import numpy as np
import sympy
from skverify import to_sympy

## 1. A first trace

Start with a function everyone knows: `scipy.integrate.trapezoid`. We all learned the trapezoid rule in school. Does scipy compute the thing we learned?

Let's ask.

In [2]:
from scipy.integrate import trapezoid

y = np.linspace(0, 1, 8) ** 2
t = to_sympy(lambda y: trapezoid(y, dx=0.1), y)
t.formula

0.05*y[0] + 0.05*y[7] + Sum(0.1*y[j + 1], (j, 0, 5))

Read it slowly. There is a `Sum` of the middle points, each weighted `0.1`. And `y[0]` and `y[7]`, the two endpoints, sit outside the Sum with half weight, `0.05`.

That is exactly the trapezoid rule from the textbook: full weight inside, half weight at the ends. The formula shows the *structure* of the rule.

And nothing was lost. The number is still there too:

In [3]:
t.value, trapezoid(y, dx=0.1)

(np.float64(0.2357142857142857), np.float64(0.2357142857142857))

*Try it yourself: swap in `simpson` or `cumulative_trapezoid` and see their structure too.*

Same number both ways. The trace gives you the formula and the value together.

## 2. Now your own code

So far we traced scipy. But the real point is tracing *your* code.

Here is a heat equation step, written by hand. The question every student asks: did I get the update rule right?

In [4]:
def step(u, dt, h):
    lap = (u[2:] - 2 * u[1:-1] + u[:-2]) / h**2
    unew = u.copy()
    unew[1:-1] = u[1:-1] + dt * lap
    return unew

s = to_sympy(step, np.sin(np.linspace(0, np.pi, 9)), 0.01, 0.1)
s.formula

Piecewise((dt*(u[i + 1] + u[i - 1] - 2*u[i])/h**2 + u[i], (i >= 1) & (i < 8)), (u[i], True))

Look at what came back.

First: `dt` and `h` stayed as letters. We passed numbers (`0.01` and `0.1`), but the tool kept them symbolic, named after our arguments. The formula is general in the parameters.

Second: the formula is a `Piecewise`. It says the interior points (`1 <= i < 8`) get the stencil, and the endpoints stay untouched. Our boundary handling is visible, not buried.

Reading is good. Proving is better. Let's write down the rule we *intended*, and ask the tool if our code equals it:

In [5]:
from skverify.checks import against
from skverify.helpers import axis_idx

i = axis_idx(0)
U = sympy.IndexedBase("u")
dt, h = sympy.symbols("dt h", real=True)

textbook = sympy.Piecewise(
    (U[i] + dt * (U[i + 1] - 2 * U[i] + U[i - 1]) / h**2, (i >= 1) & (i < 8)),
    (U[i], True),
)
against(s, textbook)

Evidence(verdict='proven', method='canonical', detail=0)

The verdict is `proven`. That word means symbolic equality: the two formulas are the same mathematics, decided by sympy, not by comparing a few lucky numbers.

Our code is right, and now we know it in the strongest sense available.

*Try it yourself: change the stencil, retrace, and watch `against` flip to `refuted` with the residual showing your change.*

## 3. Formulas are useful, not just pretty

Once your code is a formula, everything sympy can do applies to it.

Here is a hand-rolled cross-entropy loss. We will trace it, and then *differentiate the formula* to get a gradient we never wrote:

In [6]:
def loss(w, x, t):
    p = 1.0 / (1.0 + np.exp(-(x @ w)))
    return -np.sum(t * np.log(p) + (1 - t) * np.log(1 - p))

x = np.array([[1.0, 2.0], [1.0, -1.0], [1.0, 0.5]])
t = np.array([1.0, 0.0, 1.0])
L = to_sympy(loss, np.array([0.1, -0.2]), x, t)
L.formula

-Sum((1 - t[j])*log(1 - 1.0/(1.0 + exp(-Sum(w[k]*x[j, k], (k, 0, 1))))) + log(1.0/(1.0 + exp(-Sum(w[k]*x[j, k], (k, 0, 1)))))*t[j], (j, 0, 2))

In [7]:
W = sympy.IndexedBase("w")
grad0 = sympy.simplify(sympy.diff(L.formula.doit(), W[0]))

# check against finite differences
subs = {sympy.IndexedBase("x")[idx]: float(v) for idx, v in np.ndenumerate(x)}
subs |= {sympy.IndexedBase("t")[k]: float(v) for k, v in enumerate(t)}
g = sympy.lambdify((W[0], W[1]), grad0.xreplace(subs))

eps = 1e-6
fd = (loss([0.1 + eps, -0.2], x, t) - loss([0.1 - eps, -0.2], x, t)) / (2 * eps)
g(0.1, -0.2), fd

(np.float64(-0.5000000000000001), np.float64(-0.500000000069889))

*Try it yourself: differentiate with respect to `w[1]`, or take the second derivative for the Hessian entry.*

The gradient from the traced formula matches finite differences to eight decimals. We never wrote that derivative. Sympy computed it, from a formula that came from running plain numpy code.

## 4. Find a bug by subtracting formulas

Now the story this tool exists for.

I implemented Simpson's rule from a paper, and I made a typo: a weight of `3` where the rule says `4`. The value comes out wrong, but a wrong value only tells you *that* you have a bug, not *where*.

So instead of comparing values, we compare formulas. Trace my buggy version, trace scipy's correct one, and subtract:

In [8]:
from scipy.integrate import simpson

def my_simpson(y, h):
    total = y[0] + y[-1]
    for k in range(1, len(y) - 1):
        total = total + (3 if k % 2 == 1 else 2) * y[k]   # bug: 3 should be 4
    return total * h / 3

y9 = np.linspace(0, 1, 9) ** 2
mine = to_sympy(my_simpson, y9, 0.125)
ref = to_sympy(lambda y: simpson(y, dx=0.125), y9)

diff = (mine.formula - ref.formula).doit().subs(sympy.Symbol("h", real=True), sympy.Rational(1, 8))
sympy.nsimplify(sympy.expand(diff), rational=True)

-y[1]/24 - y[3]/24 - y[5]/24 - y[7]/24

Look at what survived the subtraction: only the odd indices, each with coefficient `-1/24`.

Work that coefficient backwards. The weight difference is `3 - 4 = -1`. Divide by the rule's `3`, multiply by `h = 1/8`, and you get `-1/24` exactly.

So the subtraction did not just say "wrong". It said: *your bug is in the odd branch, and your weight is off by exactly one.* That is a bug report written in mathematics.

## 5. The certificate is honest about itself

One more thing, and it matters most.

When your code branches on the data, the formula is only valid for inputs that take the same branch. The tool does not hide this. It records the branch condition and hands it to you as a precondition:

In [9]:
def model(u):
    if u.max() > 2.0:
        return u * 0.5
    return u * 2.0

r = to_sympy(model, np.array([1.0, 3.0]))
r.formula, r.preconditions

(0.5*u[i], Max(u[0], u[1]) > 2.0)

The formula came with its condition: this rule holds when `Max(u[0], u[1]) > 2.0`, which is the branch our data took.

And when the trace meets *compiled* code it cannot enter, like the banded solver inside a scipy spline, it does not guess. The compiled result becomes a named box in the formula, and the trace checks what it can about that box:

In [10]:
from scipy.interpolate import make_interp_spline

def fit(x, y):
    return make_interp_spline(x, y)

x8 = np.linspace(0, 7, 8)
spl = to_sympy(fit, x8, np.sin(x8))
spl.c[2].formula

dgbsv_1_2[2, 0]

In [11]:
# the banded solve is a COMPILED routine: it becomes a named atom,
# and its residual A @ x == b was checked on this very call
[(entry[0], dict(entry[1])) for entry in spl.unchecked]

[('coloc', {'contract': 'unknown'}), ('dgbsv', {'residual': 'ok'})]

## 6. Watch the whole computation: `derivation()`

The formula is the *end* of the story. Sometimes you want the story itself: every intermediate step, in order.

`derivation()` prints exactly that. Shared subexpressions get names (`t0`, `t1`) so each line stays short:

In [12]:
def rms(u):
    return np.sqrt(np.sum(u * u) / len(u))

r = to_sympy(rms, np.array([3.0, 4.0]))
print(r.derivation())

step 0: u[i]
step 1: u[i]**2
step 3: Sum(u[j]**2, (j, 0, 1))
step 4: step[3]/2
step 5: sqrt(2)*sqrt(step[3])/2
result: sqrt(2)*sqrt(step[3])/2


Read it top to bottom: the input arrives, gets squared, summed, divided, rooted. The last line is always the result. For big traces (a full scipy spline is thousands of steps) repeated loop bodies fold into one rule line, so the printout follows the *algorithm's* size, not the data's.

## 7. Real libraries, real learned models

Everything so far also works on code you did not write.

Statsmodels: trace an ordinary least squares fit and get the slope as a formula of your data:

In [13]:
from statsmodels.api import OLS

def fit_slope(y, X):
    return OLS(y, X).fit().params

n = 6
X = np.column_stack([np.ones(n), np.arange(float(n))])
yobs = X @ np.array([2.0, 0.5]) + 0.01 * np.arange(n)
params = to_sympy(fit_slope, yobs, X)
print(str(np.atleast_1d(params)[1].formula)[:120], "...")

Sum(Piecewise((y[k2]*Sum(1.0*svd_0_0[k2, k]*svd_0_2[k, 1]/svd_0_1[1], (k, 0, 1)), Eq(k, 1)), (y[k2]*Sum(1.0*svd_0_0[k2,  ...


The slope is a formula in `y` and `X`, built through statsmodels' own pinv path. The `svd_0` names are the compiled decomposition inside, disclosed as named boxes.

Sklearn, with a twist: take a *fitted* SVM and trace its decision function through sklearn's own kernel. Then compare with what the C library libsvm answers:

In [14]:
from sklearn.svm import SVC
from sklearn.metrics.pairwise import rbf_kernel

rng = np.random.default_rng(4)
Xtr = rng.standard_normal((20, 2))
ycls = (Xtr @ np.array([1.0, -0.5]) > 0).astype(int)
svc = SVC(kernel="rbf").fit(Xtr, ycls)

SV, alpha = svc.support_vectors_, svc.dual_coef_.ravel()
b, gamma = float(svc.intercept_[0]), svc._gamma

def decision(x):
    return rbf_kernel(x, SV, gamma=gamma) @ alpha + b

xq = np.array([[0.5, -1.0], [1.5, 0.3]])
d = to_sympy(decision, xq)
np.allclose(np.asarray(d.value, dtype=float), svc.decision_function(xq))

True

`True`: the textbook kernel expansion, traced symbolically, reproduces the compiled C answer exactly. We just used the tool to *check libsvm itself*.

## 8. The top of the ladder: check code against theory

The strongest thing you can do: compare your code not with other code, but with mathematics.

Trace a hand-written normal density. The parameters stay symbolic, so we can ask sympy.stats whether our code *is* the normal distribution:

In [15]:
from sympy.stats import Normal, density

def normal_pdf(x, mu, sigma):
    z = (x - mu) / sigma
    return np.exp(-0.5 * z * z) / (sigma * np.sqrt(2.0 * np.pi))

p = to_sympy(normal_pdf, np.linspace(-2, 3, 8), 0.5, 1.3)

t = sympy.Symbol("t", real=True)
mine = p.formula.subs(sympy.IndexedBase("x")[axis_i], t) if False else None
from skverify.helpers import axis_idx
mine = p.formula.subs(sympy.IndexedBase("x")[axis_idx(0)], t)

# exactify the float constant 0.3989... back to 1/sqrt(2 pi)
for f in mine.atoms(sympy.Float):
    cand = sympy.nsimplify(f, [sympy.pi])
    if abs(float(cand) - float(f)) < 1e-12:
        mine = mine.subs(f, cand)

mu = sympy.Symbol("mu", real=True)
sig = sympy.Symbol("sigma", positive=True)
theory = density(Normal("N", mu, sig))(t)
sympy.simplify(mine.subs(sympy.Symbol("sigma", real=True), sig) - theory)

0

Zero. Our numpy code and the textbook density are the *same expression*, proven symbolically, parameters and all.

And because it is a real density we can even integrate it:

In [16]:
sympy.integrate(mine.subs(sympy.Symbol("sigma", real=True), sig), (t, -sympy.oo, sympy.oo))

1

It normalizes to 1 over the whole line. Code, checked against theory, by machine.

---

## Where to go next

### Try it on your own code

The recipe is always the same three lines:

```python
from skverify import to_sympy
result = to_sympy(my_function, my_arrays)
result.formula
```

Good first candidates: any function that takes arrays in and returns arrays or numbers out. Loops, slices, masks and scipy calls inside are fine.

### Look deeper into a result

Every traced result carries more than the formula:

```python
result.value            # the concrete answer, always exact
result.preconditions    # the conditions this formula assumes
result.unchecked        # compiled results the trace disclosed
print(result.derivation())   # the whole computation, step by step
```

### Let sympy work for you

The formula is ordinary sympy, so everything composes:

```python
sympy.diff(result.formula, x)     # gradients you never wrote
sympy.latex(result.formula)       # straight into your paper
sympy.simplify, .subs, lambdify   # the whole toolbox
```

### Check code against mathematics

```python
from skverify.checks import against, banded
against(result, textbook_expression)   # proven / refuted / unknown
```

### What works today

Real code from **scipy** (splines, quadrature, statistics), **statsmodels** (regression, autocovariance, diagnostics) and **sklearn** (metrics, kernels, SVM decision functions). When a trace cannot pass a wall it refuses loudly and tells you why; it never guesses.

Found something that should trace but does not? That is exactly the feedback we want: open an issue with your function.